# Robustez de Modelos: Ataques Adversariales y Defensas

## Vulnerabilidades de Modelos de IA

Los modelos de IA pueden ser engañados mediante **ataques adversariales**: perturbaciones mínimas
que provocan clasificaciones incorrectas. Este cuaderno simula ataques comunes y defiende modelos.

### Tipos de ataques
| Ataque | Técnica | Riesgo |
| :--- | :--- | :--- |
| **Evasión** | Perturbación de características | Malware se clasifica como benigno |
| **Envenenamiento** | Inyección de datos en entrenamiento | Modelo aprende comportamiento anómalo |
| **Extracción** | Robo del modelo mediante queries | Réplica no autorizada |
| **Inversión** | Reconstrucción de datos de entrenamiento | Fuga de privacidad |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import ks_2samp

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("Importaciones OK")

## 1. Preparar dataset y modelo base

In [ ]:
def generar_dataset_malware(n_benign=1700, n_malicious=300, seed=42):
    rng = np.random.default_rng(seed)
    benign = pd.DataFrame({
        "entry_point":       rng.integers(4096, 16384,   size=n_benign).astype(float),
        "num_sections":      rng.integers(3, 7,           size=n_benign).astype(float),
        "entropia_max":      rng.uniform(4.0, 6.5,        size=n_benign),
        "num_importaciones": rng.integers(20, 150,        size=n_benign).astype(float),
        "num_dlls":          rng.integers(2, 12,          size=n_benign).astype(float),
        "file_size":         rng.integers(50_000, 2_000_000, size=n_benign).astype(float),
        "has_debug":         rng.integers(0, 2,           size=n_benign).astype(float),
        "label":             0
    })
    malicious = pd.DataFrame({
        "entry_point":       rng.integers(0, 4096,        size=n_malicious).astype(float),
        "num_sections":      rng.integers(8, 20,          size=n_malicious).astype(float),
        "entropia_max":      rng.uniform(6.8, 8.0,        size=n_malicious),
        "num_importaciones": rng.integers(200, 600,       size=n_malicious).astype(float),
        "num_dlls":          rng.integers(15, 40,         size=n_malicious).astype(float),
        "file_size":         rng.integers(10_000, 500_000, size=n_malicious).astype(float),
        "has_debug":         np.zeros(n_malicious, dtype=float),
        "label":             1
    })
    return pd.concat([benign, malicious], ignore_index=True).sample(frac=1, random_state=seed)

FEATURES = ["entry_point", "num_sections", "entropia_max",
            "num_importaciones", "num_dlls", "file_size", "has_debug"]

df = generar_dataset_malware()
X, y = df[FEATURES].values, df["label"].values

# Normalizar para facilitar la perturbación proporcional
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, stratify=y, random_state=SEED
)

# Modelo base
modelo_base = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
modelo_base.fit(X_train, y_train)
acc_base = accuracy_score(y_test, modelo_base.predict(X_test))
f1_base  = f1_score(y_test, modelo_base.predict(X_test))
print(f"Modelo base — Accuracy: {acc_base:.4f} | F1: {f1_base:.4f}")

## 2. Ataque 1 — Ruido gaussiano (evasión)

Añadimos ruido gaussiano pequeño a las características del conjunto de prueba.
Simula que un atacante modifica levemente el malware para evadir la detección.

In [ ]:
def ataque_ruido_gaussiano(X: np.ndarray, sigma: float, seed: int = 42) -> np.ndarray:
    """Añade ruido gaussiano y recorta el resultado al rango [0,1]."""
    rng = np.random.default_rng(seed)
    return np.clip(X + rng.normal(0, sigma, X.shape), 0, 1)

sigmas = [0.01, 0.03, 0.05, 0.10, 0.20]
resultados_ruido = []

for sigma in sigmas:
    X_adv = ataque_ruido_gaussiano(X_test, sigma)
    acc = accuracy_score(y_test, modelo_base.predict(X_adv))
    f1  = f1_score(y_test,         modelo_base.predict(X_adv))
    resultados_ruido.append({"sigma": sigma, "accuracy": acc, "f1": f1})
    print(f"sigma={sigma:.2f} → Accuracy={acc:.4f} | F1={f1:.4f}")

df_ruido = pd.DataFrame(resultados_ruido)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_ruido.sigma, df_ruido.accuracy, "o-", label="Accuracy", color="steelblue")
ax.plot(df_ruido.sigma, df_ruido.f1,       "s-", label="F1",       color="darkorange")
ax.axhline(acc_base, ls="--", color="steelblue", alpha=0.5, label="Accuracy base")
ax.axhline(f1_base,  ls="--", color="darkorange", alpha=0.5, label="F1 base")
ax.set_xlabel("Sigma del ruido gaussiano")
ax.set_ylabel("Métrica")
ax.set_title("Impacto del ruido adversarial en el modelo base")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Ataque 2 — Evasión por movimiento de características

Un atacante que conoce el modelo puede reducir características discriminantes
(e.g., bajar la entropía, reducir las importaciones) para que el malware
parezca benigno.

In [ ]:
def ataque_evasion_caracteristicas(X: np.ndarray,
                                    y: np.ndarray,
                                    indices_objetivo: list,
                                    reduccion: float = 0.3) -> np.ndarray:
    """
    Solo perturba muestras maliciosas (y==1).
    Reduce características en 'indices_objetivo' un porcentaje 'reduccion'.
    """
    X_adv = X.copy()
    mask_mal = y == 1
    for idx in indices_objetivo:
        X_adv[mask_mal, idx] *= (1 - reduccion)
    return X_adv

# Características más importantes según el modelo: índices de
# entropia_max (2) y num_importaciones (3)
idx_entropia      = FEATURES.index("entropia_max")
idx_importaciones = FEATURES.index("num_importaciones")

reducciones = [0.1, 0.2, 0.3, 0.5, 0.7]
resultados_evasion = []

for r in reducciones:
    X_adv = ataque_evasion_caracteristicas(
        X_test, y_test, [idx_entropia, idx_importaciones], reduccion=r
    )
    acc = accuracy_score(y_test, modelo_base.predict(X_adv))
    f1  = f1_score(y_test, modelo_base.predict(X_adv))
    resultados_evasion.append({"reduccion": r, "accuracy": acc, "f1": f1})
    print(f"reduccion={r:.1f} → Accuracy={acc:.4f} | F1={f1:.4f}")

df_evasion = pd.DataFrame(resultados_evasion)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_evasion.reduccion, df_evasion.f1, "o-", color="tomato", lw=2)
ax.axhline(f1_base, ls="--", color="gray", label=f"F1 base = {f1_base:.3f}")
ax.set_xlabel("Reducción aplicada a características clave")
ax.set_ylabel("F1 Score")
ax.set_title("Impacto de evasión por reducción de características")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Defensa — Entrenamiento adversarial

Reentrenamos el modelo incluyendo ejemplos adversariales en el conjunto
de entrenamiento. Esto hace al modelo más robusto frente a perturbaciones.

In [ ]:
# Generar ejemplos adversariales del conjunto de entrenamiento
X_train_adv_ruido = ataque_ruido_gaussiano(X_train, sigma=0.05, seed=SEED+1)
X_train_adv_evas  = ataque_evasion_caracteristicas(
    X_train, y_train, [idx_entropia, idx_importaciones], reduccion=0.3
)

# Dataset de entrenamiento robusto = original + adversariales
X_robusto = np.vstack([X_train, X_train_adv_ruido, X_train_adv_evas])
y_robusto = np.concatenate([y_train, y_train, y_train])

print(f"Tamaño entrenamiento original : {len(X_train)}")
print(f"Tamaño entrenamiento robusto  : {len(X_robusto)}")

# Entrenar modelo robusto
modelo_robusto = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=SEED)
modelo_robusto.fit(X_robusto, y_robusto)

print("\nComparación de modelos en diferentes escenarios de ataque:")
print(f"{'Escenario':<30} {'Base F1':>10} {'Robusto F1':>12}")
print("-" * 54)

escenarios = {
    "Sin ataque"             : X_test,
    "Ruido σ=0.05"           : ataque_ruido_gaussiano(X_test, 0.05),
    "Ruido σ=0.10"           : ataque_ruido_gaussiano(X_test, 0.10),
    "Evasión 30%"            : ataque_evasion_caracteristicas(X_test, y_test, [idx_entropia, idx_importaciones]),
}

for nombre, X_esc in escenarios.items():
    f1_b = f1_score(y_test, modelo_base.predict(X_esc))
    f1_r = f1_score(y_test, modelo_robusto.predict(X_esc))
    mejor = "<-- mejora" if f1_r > f1_b else ""
    print(f"{nombre:<30} {f1_b:>10.4f} {f1_r:>12.4f}  {mejor}")

## 5. Detección de deriva de datos (data drift)

Detectamos si la distribución de los datos de entrada en producción difiere
significativamente de la distribución de referencia, usando el test **Kolmogorov-Smirnov**.

In [ ]:
def detectar_deriva_ks(X_ref: np.ndarray, X_prod: np.ndarray,
                        feature_names: list, umbral_p: float = 0.05):
    """Test KS por característica para detectar deriva de datos."""
    resultados = []
    for i, fname in enumerate(feature_names):
        stat, p = ks_2samp(X_ref[:, i], X_prod[:, i])
        resultados.append({
            "caracteristica": fname,
            "ks_stat": round(stat, 4),
            "p_valor": round(p, 6),
            "deriva":  p < umbral_p
        })
    return pd.DataFrame(resultados)

# Simular datos de producción atacados (con ruido moderado)
X_prod_atacado = ataque_ruido_gaussiano(X_test, sigma=0.15)

resultado_ks = detectar_deriva_ks(X_train, X_prod_atacado, FEATURES)
print("Test de Kolmogorov-Smirnov — Deriva de datos:")
print(resultado_ks.to_string(index=False))

# Visualizar
fig, ax = plt.subplots(figsize=(9, 4))
colores = ["tomato" if d else "steelblue" for d in resultado_ks.deriva]
ax.barh(resultado_ks.caracteristica, resultado_ks.ks_stat, color=colores)
ax.axvline(0.1, ls="--", color="gray", label="Umbral indicativo")
ax.set_xlabel("Estadístico KS")
ax.set_title("Detección de deriva — Rojo = deriva significativa")
ax.legend()
plt.tight_layout()
plt.show()

print("\n✓ Cuaderno 7 completado.")